# V12: 정예 피처 구성 및 석식 0명 패턴 집중 방어

이 노트북은 과적합을 방지하기 위해 피처를 정예화하고, 석식 참여율이 0에 가까워지는 '수요일/금요일' 패턴을 집중 방어하여 60점대 진입을 목표로 합니다.

## 주요 전략
1. **피처 다이어트**: 날씨(기온, 비온날만), 메뉴(중식 고기메뉴 수만) 핵심 피처로 압축
2. **요일별 석식 보정**: 수요일(자기계발의 날)과 금요일(조기 퇴근) 석식 참여율 급감 집중 반영
3. **시간외근무 핵심 활용**: 석식계 예측의 결정적 변수인 시간외근무 승인건수 비중 강화
4. **CatBoost 단일 고도화**: 앙상블보다 과적합 방지에 유리한 CatBoost 단일 모델(혹은 고비중) 전략

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from catboost import CatBoostRegressor
except ImportError:
    CatBoostRegressor = None
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from korean_font import set_korean_font
import warnings

set_korean_font()
warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

한글 폰트 설정: Malgun Gothic (c:/Windows/Fonts/malgun.ttf)


In [2]:
# 1. 데이터 로드
train = pd.read_csv('data/train_median.csv')
test = pd.read_csv('data/test.csv')
weather_train = pd.read_csv('data/weather.csv')
weather_test = pd.read_csv('data/weatherTest.csv')

# 2. 날씨 데이터 정제 (기온, 비온날만 사용)
weather = pd.concat([weather_train, weather_test], axis=0)
weather['일시'] = pd.to_datetime(weather['일시'])
weather['비온날'] = (weather['강수량'].fillna(0) > 0).astype(int)
weather = weather[['일시', '기온', '비온날']]

# 3. 데이터 결합
train['일자'] = pd.to_datetime(train['일자'])
test['일자'] = pd.to_datetime(test['일자'])
train = pd.merge(train, weather, left_on='일자', right_on='일시', how='left')
test = pd.merge(test, weather, left_on='일자', right_on='일시', how='left')
train.drop(['일시'], axis=1, inplace=True)
test.drop(['일시'], axis=1, inplace=True)

print("데이터 결합 완료")

데이터 결합 완료


In [3]:
# 4. 정예 피처 엔지니어링
def refined_engineering(df):
    # 날짜
    df['월'] = df['일자'].dt.month
    df['요일'] = df['일자'].dt.weekday
    df['일'] = df['일자'].dt.day
    
    # 식사가능자수 (강력 피처)
    df['식사가능자수'] = df['본사정원수'] - (df['본사휴가자수'] + df['본사출장자수'] + df['현본사소속재택근무자수'])
    
    # 공휴일 전후
    df = df.sort_values('일자')
    df['공휴일전날'] = (df['일자'].shift(-1).diff().dt.days > 2).astype(int)
    df['공휴일후날'] = (df['일자'].diff().dt.days > 2).astype(int)
    
    # 중식 고기메뉴 수 (간결하게)
    meat_list = ['돈까스', '제육', '불고기', '고기', '치킨', '스테이크', '갈비']
    df['중식_고기메뉴'] = df['중식메뉴'].apply(lambda x: sum(1 for m in meat_list if m in str(x)))
    
    # 석식 수요일/금요일 구분
    df['수요일석식'] = (df['요일'] == 2).astype(int)
    df['금요일석식'] = (df['요일'] == 4).astype(int)
    
    return df.sort_index().fillna(0)

train = refined_engineering(train)
test = refined_engineering(test)

train['중식참여율'] = train['중식계'] / train['식사가능자수']
train['석식참여율'] = train['석식계'] / train['식사가능자수']

print("피처 정제 완료")

피처 정제 완료


In [4]:
# 5. 모델링: 중식과 석식의 분리 전략
lunch_features = ['월', '요일', '일', '식사가능자수', '본사출장자수', '기온', '비온날', '공휴일전날', '중식_고기메뉴']
dinner_features = ['월', '요일', '일', '식사가능자수', '본사시간외근무명령서승인건수', '기온', '비온날', '공휴일전날', '공휴일후날', '수요일석식', '금요일석식']

def train_best_model(x_tr, y_tr, x_te, name):
    print(f"{name} 모델 학습 중...")
    if CatBoostRegressor:
        model = CatBoostRegressor(n_estimators=3000, learning_rate=0.02, depth=6, random_state=42, verbose=0)
    else:
        model = XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6, random_state=42)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)

pred_lunch_ratio = train_best_model(train[lunch_features], train['중식참여율'], test[lunch_features], "중식")
pred_dinner_ratio = train_best_model(train[dinner_features], train['석식참여율'], test[dinner_features], "석식")

# 결과 복원 및 매핑
test['pred_lunch'] = pred_lunch_ratio * test['식사가능자수']
test['pred_dinner'] = pred_dinner_ratio * test['식사가능자수']

sample_sub = pd.read_csv('data/sample_submission.csv')
test['일자_str'] = test['일자'].dt.strftime('%Y-%m-%d')
final_sub = pd.merge(sample_sub[['일자']], test[['일자_str', 'pred_lunch', 'pred_dinner']], 
                     left_on='일자', right_on='일자_str', how='left')
final_sub = final_sub[['일자', 'pred_lunch', 'pred_dinner']]
final_sub.columns = ['일자', '중식계', '석식계']

final_sub['중식계'] = final_sub['중식계'].round(0).astype(int)
final_sub['석식계'] = final_sub['석식계'].round(0).astype(int)

final_sub.to_csv('submission/submission_v12.csv', index=False, encoding='utf-8-sig')
print("V12 저장 완료: submission/submission_v12.csv")

중식 모델 학습 중...
석식 모델 학습 중...
V12 저장 완료: submission/submission_v12.csv
